In [2]:
import geopandas as gpd 
import numpy as np 
import pandas as pd 
import sys 
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from functions.funcs import *

In [5]:
ds = gpd.read_parquet(r"..\Data\Mapped_SAT_MI_Cleanedspeeds.parquet")

In [6]:

target_date = pd.to_datetime("2024-01-01") ## picks dFAD locations one day after this date 
print(target_date)

ds_active = querry_date(ds, date = target_date) ## All of the active dFADs at this time 
ds_active = ds_active.reset_index()
columns = ["TimeStamp", "x_speed", "y_speed"]
ds_locations = pd.DataFrame()
for label in columns: 
    longlist, ids = Column_to_List(ds_active, label, idlist = True)
    
    ds_locations[label] = longlist
lat, lon  = list_of_latlon(ds_active, droplast= False)
ds_locations["lat"] = lat
ds_locations["lon"] =lon
ds_locations["BuoyName"] = ids
ds_locations.TimeStamp = pd.to_datetime(ds_locations.TimeStamp)

##Filter Timestep by certain threshhold to get locations of FADS within closes  
## UPDATE:This might be better to interp these onto the specific time. 
hourlim = 24
time_threshhold  = pd.Timedelta(hours= hourlim)
time_upper  = target_date + time_threshhold ## This is set for dFADs one day after the date 
time_lower = target_date 
ds_locations = ds_locations.query(f"TimeStamp > @time_lower")
ds_locations = ds_locations.query(f"TimeStamp < @time_upper")
print(f"Amount of sampled dFAD within {hourlim} hrs : {len(ds_locations)}")
ds_locations = ds_locations.drop_duplicates(subset=["lat"], keep="first")
ds_locations = ds_locations.drop_duplicates(subset=["lon"], keep="first").reset_index(drop = True)

## New get only the first point of the day for the forcast.

dFADs = ds_locations.sort_values('TimeStamp').groupby("BuoyName").first()
print(f"Number of Unique dFADs/ points avalable: {len(dFADs)}")
# if len(dFADs) < 1: 
#     continue
dFADs = dFADs.reset_index()

2024-01-01 00:00:00
Amount of sampled dFAD within 24 hrs : 106
Number of Unique dFADs/ points avalable: 15


In [ ]:
# Create the persistence model
def persistence_model(merged_df, start_time, window_hrs, max_horizon_hrs=None, define_persistence_as='mean', velocity_window=2, dt_freq=None):
    df = merged_df.copy()

    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df=df.sort_values(['DateTime']).reset_index(drop=True)
 
    # Set start time as the start time of true trajectory
    if start_time is None:
        if len(df)>=velocity_window:
            start_time=df['DateTime'].iloc[velocity_window-1]
        
        else:
            start_time=df['DateTime'].iloc[0]
        
    start_time = pd.to_datetime(start_time)
 
    # Base row
    row_t0 = df.loc[(df['DateTime'] ==start_time)]
    lat_t0 = row_t0["Latitude_true"].values
    lon_t0 = row_t0["Longitude_true"].values
     
    # Persistence base
    if window_hrs <= 0:
        base = row_t0[['Latitude_true', 'Longitude_true', 'speed_ms_true']].iloc[0].to_dict()
    else:
        t_min = start_time - pd.Timedelta(hours=window_hrs)
        past = df[(df['DateTime'] > t_min) & (df['DateTime'] <= start_time)]
        if past.empty:
            past = row_t0
        if define_persistence_as == 'last':
            base_row = past.sort_values('DateTime').iloc[-1]
        elif define_persistence_as == 'median':
            base_row = past.median()
        else:
            base_row = past.mean()
        base = {
            'Latitude_true': base_row['Latitude_true'],
            'Longitude_true': base_row['Longitude_true'],
            'speed_ms_true': base_row['speed_ms_true']
        }
    
    # Velocity
    past_window = df[df['DateTime'] <= start_time].sort_values('DateTime').tail(velocity_window)
    if len(past_window) >= 2:
        dt_seconds = (past_window['DateTime'].iloc[-1] - past_window['DateTime'].iloc[0]).total_seconds()
        if dt_seconds>0:
            v_latitude = (past_window['Latitude_true'].iloc[-1] - past_window['Latitude_true'].iloc[0]) / dt_seconds
            v_longitude = (past_window['Longitude_true'].iloc[-1] - past_window['Longitude_true'].iloc[0]) / dt_seconds
        else:
            v_latitude = 0
            v_longitude = 0
    else:
        v_latitude = 0
        v_longitude = 0
    
    if dt_freq is None:
        dt=df['DateTime'].diff().median()
        if pd.isna(dt):
            dt=pd.Timedelta(hours=1)
        freq=pd.Timedelta(dt).to_pytimedelta()
 
        end_time=start_time+pd.Timedelta(hours=max_horizon_hrs)
        future_times=pd.date_range(start=start_time, end=end_time, freq=dt)
    else:
        end_time=start_time+pd.Timedelta(hours=max_horizon_hrs)
        future_times=pd.date_range(start=start_time, end=end_time, freq=dt_freq)

    end_time=start_time+pd.Timedelta(hours=max_horizon_hrs)
    ### Setting to Future points 
    future_points = df[(df['DateTime'] < end_time) & (df['DateTime'] >= start_time)]
    future_times = future_points["DateTime"].to_list()
    future=pd.DataFrame({'DateTime': future_times})
    dt_future=(future['DateTime']-start_time).dt.total_seconds()

    # Extrapolate
    future['Latitude_persistence'] = lat_t0 + v_latitude * dt_future
    future['Longitude_persistence'] = lon_t0 + v_longitude * dt_future
    future['speed_ms_persistence'] = base['speed_ms_true']
    future['lead_time_hours']=dt_future/3600.0
    return future, pd.to_datetime(start_time)

In [134]:
## Puting things into Persistance model
BuoyID = dFADs.at[2, "BuoyName" ]
print(BuoyID)
target_time = dFADs.at[2, "TimeStamp" ]
Time = dFADs.at[2,"TimeStamp"]
dFAD = ds.query(f"BuoyName == @BuoyID").reset_index(drop = True)
speed = dFAD.xy_speed.values[0]
times = dFAD.TimeStamp.values[0]
lat,lon = list_of_latlon(dFAD, False)
onedFAD = pd.DataFrame({"DateTime" : times, "Latitude_true": lat, "Longitude_true": lon, "speed_ms_true": speed})

future, time = persistence_model(onedFAD, target_time, 10,8*24, velocity_window = 2, dt_freq= pd.Timedelta(hours= 4), )
print(len(future))


M3i+127312
21


               DateTime  Latitude_true  Longitude_true  speed_ms_true
0   2023-12-29 05:59:00         4.6578       -163.6510       0.344033
1   2023-12-29 10:00:00         4.6245       -163.5928       0.510547
2   2023-12-29 13:58:00         4.5852       -163.5475       0.458892
3   2023-12-29 17:59:00         4.5475       -163.5050       0.429065
4   2023-12-31 05:59:00         4.5993       -163.4253       0.080568
..                  ...            ...             ...            ...
70  2024-01-15 17:59:00         4.7810       -161.1415       0.445716
71  2024-01-15 21:59:00         4.8080       -161.1128       0.296706
72  2024-01-16 01:58:00         4.8372       -161.0657       0.423402
73  2024-01-16 05:59:00         4.8602       -161.0230       0.368508
74  2024-01-16 09:59:00         4.8592       -160.9950       0.216336

[75 rows x 4 columns]


In [9]:
Forcast_times = pd.read_csv(r"saved_output\combined_krigging2024.csv")
Forcast_times = Forcast_times[Forcast_times.isna().any(axis=1)].reset_index()

In [118]:
def Forcast_snippit(ds: gpd.GeoDataFrame, dates, startdate, length)-> gpd.GeoDataFrame: 
    """Grad only the snipbit of dFAD trajectory that lines up with forcast window"""
    ds_s = ds.copy()
    forecast_end = startdate + length
    for i in range(len(ds)): ## Try and grab at the exact start times from dates they should be the same
        timelist = (ds_s.at[i,"TimeStamp"])
        timelist = pd.to_datetime(timelist)
        mask = (timelist >=startdate) & (timelist <= forecast_end)
        timelist = timelist[mask]
        coords = np.asarray(ds.at[i,"geometry"].coords)
        filtered_coords = coords[mask]
        ds_s.at[i,"TimeStamp"] = timelist
        if len(filtered_coords) > 1:
            ds_s.at[i,"geometry"] = sp.geometry.LineString(filtered_coords)
        else: 
            ds_s.at[i,"geometry"] = None
    return ds_s


In [ ]:
buoy_list = dFADs.BuoyName.tolist()
ds_filtered = ds_active[ds_active["BuoyName"].isin(buoy_list)].reset_index(drop = True)
ds_short_t = Forcast_snippit(ds_filtered, dFADs.TimeStamp, target_date, pd.Timedelta(days = 8))
target = ds_short_t.query("BuoyName == @BuoyID")

ds_short_t



,level_0,index,BuoyName,MinOfDate,MaxOfDate,TimeStamp,geometry,x_deg,y_deg,x_km,...,x_speed,y_speed,xy_speed,points_removed,Masked_array,points_removed2,Masked_array2,Masked_array_combined,mapped_v,mapped_u
0,1572,1700,SLX+418929,2023-10-31 11:31:00,2024-01-02 15:31:00,"DatetimeIndex(['2024-01-01 03:30:00', '2024-01...","LINESTRING (-161.37682 5.80588, -161.25913 5.8...","[0.005059999999986076, 0.02610000000001378, 0....","[0.002489999999999881, 0.004700000000000593, -...","[0.5626463288195196, 2.902187585425462, 1.4733...",...,"[0.29304496292683313, 0.23254708216550174, 0.1...","[0.13838802662510324, 0.04018578266052796, -0....","[0.32407730652806804, 0.23599327708464343, 0.2...",1,"[True, True, True, True, True, True, True, Tru...",60,"[False, True, False, True, False, True, False,...","[False, True, False, True, False, True, False,...","[-0.14201695117126337, -0.13574979597910455, -...","[0.38292133361641467, 0.3802578455183076, 0.35..."
1,1661,1792,M3i+127203,2023-12-23 13:00:00,2024-05-08 17:00:00,"DatetimeIndex(['2024-01-01 01:00:00', '2024-01...","LINESTRING (-161.2795 5.3697, -161.1542 5.3835...","[0.03130000000001587, 0.033999999999991815, 0....","[0.040300000000000225, 0.0519999999999996, 0.0...","[3.4804012039745285, 3.7806275059164602, 1.945...",...,"[0.2416945280537867, 0.2625435767997542, 0.134...","[0.2981782199086042, 0.3846832624697084, 0.363...","[0.38381243009770194, 0.4657081138887561, 0.38...",0,"[True, True, True, True, True, True, True, Tru...",0,"[True, True, True, True, True, True, True, Tru...","[True, True, True, True, True, True, True, Tru...","[0.12481212068451526, 0.12235672787814643, 0.1...","[0.4789833851938421, 0.5082215352234875, 0.532..."
2,1662,1793,M3i+127139,2023-12-23 21:59:00,2024-05-09 10:01:00,"DatetimeIndex(['2024-01-01 01:59:00', '2024-01...","LINESTRING (-161.2272 5.108, -161.125 5.1117, ...","[0.007499999999993179, 0.01279999999999859, 0....","[0.04789999999999939, 0.053600000000000314, 0....","[0.833961949832724, 1.423295061050603, 3.05786...",...,"[0.057673717139192525, 0.09883993479518076, 0....","[0.3522682704852817, 0.3958140729004578, 0.398...","[0.35695129705944895, 0.4079551592962548, 0.45...",0,"[True, True, True, True, True, True, True, Tru...",0,"[True, True, True, True, True, True, True, Tru...","[True, True, True, True, True, True, True, Tru...","[0.06868769141265701, 0.05704258663165117, 0.0...","[0.4633315658793225, 0.48801135530884787, 0.50..."
3,1668,1799,SLX+438132,2023-12-27 19:50:00,2024-01-07 23:50:00,"DatetimeIndex(['2024-01-01 03:50:00', '2024-01...","LINESTRING (-162.10613 5.02678, -162.01123 5.0...","[0.0039899999999875035, 0.07947999999998956, 0...","[0.0009099999999993003, 0.000600000000000378, ...","[0.443667757311892, 8.837772769706692, 8.46415...",...,"[0.6162052184887389, 0.6460360211773898, 0.587...","[0.1347066126792768, 0.004674510898791245, -0....","[0.6307569793017442, 0.6460529256025023, 0.592...",0,"[True, True, True, True, True, True, True, Tru...",0,"[True, True, True, True, True, True, True, Tru...","[True, True, True, True, True, True, True, Tru...","[0.030519455439135, -0.006888547604466089, -0....","[0.6238167738100776, 0.6353383629496775, 0.632..."
4,1671,1802,M3i+127312,2023-12-29 01:58:00,2024-01-16 09:59:00,"DatetimeIndex(['2024-01-01 01:58:00', '2024-01...","LINESTRING (-163.0978 4.7777, -163.0097 4.7832...","[0.03350000000000364, 0.05819999999999936, 0.0...","[-0.030899999999999928, -0.03329999999999966, ...","[3.7250300425950322, 6.471544730713708, 5.0371...",...,"[0.2576092698890064, 0.44754804500094797, 0.35...","[-0.22804677017639288, -0.24571703802182637, -...","[0.3440333688873562, 0.5105468671741727, 0.458...",0,"[True, True, True, True, True, True, True, Tru...",0,"[True, True, True, True, True, True, True, Tru...","[True, True, True, True, True, True, True, Tru...","[-0.09955497399670982, -0.09495718343779733, -...","[0.5204531904494865, 0.48242972776624854, 0.44..."
5,1672,1803,SLX+457829,2023-12-29 05:29:00,2024-03-27 01: